In [ ]:
!pip install pandas numpy torch scikit-learn

In [7]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

#Пути к файлам

train_path = 'train.csv'
test_path = 'test_features.csv'
output_filename = 'task1_predictions.csv'

print(f"Проверка наличия файлов:")
print(f"Train exists: {os.path.exists(train_path)}")
print(f"Test exists: {os.path.exists(test_path)}")

if not os.path.exists(train_path) or not os.path.exists(test_path):
    print("\nОШИБКА: Файлы не найдены!")
else:
    print("Файлы найдены! Начинаем работу...\n")

    #2. Загрузка данных
    print("Загрузка данных...")
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)

    print(f"Train shape: {df_train.shape}")
    print(f"Test shape: {df_test.shape}")

    target_col = 'prediction'
    id_col = 'patient_id'

    #Фильтрация колонок
    feature_cols = [col for col in df_train.columns if col not in [id_col, target_col]]
    feature_cols = [col for col in feature_cols if col in df_test.columns]

    X_train = df_train[feature_cols].copy()
    y_train = df_train[target_col].copy()
    X_test = df_test[feature_cols].copy()
    test_ids = df_test[id_col].copy()

    #3. Предобработка

    print("Предобработка...")

    for col in X_train.columns:
        if X_train[col].isnull().any():
            median_val = X_train[col].median()
            #Исправление: используем прямое присваивание вместо inplace
            X_train[col] = X_train[col].fillna(median_val)
            X_test[col] = X_test[col].fillna(median_val)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Тензоры
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    #Разделение на train/val
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_tensor, y_train_tensor, test_size=0.2, random_state=42, stratify=y_train_tensor
    )

    batch_size = 32
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

    #Для теста создаем даталоадер
    test_dataset = TensorDataset(X_test_tensor)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    #4. Модель
    input_dim = X_train.shape[1]

    class BreastCancerFFN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, 64)
            self.fc2 = nn.Linear(64, 32)
            self.fc3 = nn.Linear(32, 1)
            self.relu = nn.ReLU()
            self.dropout = nn.Dropout(0.2)

        def forward(self, x):
            x = self.relu(self.fc1(x))
            x = self.dropout(x)
            x = self.relu(self.fc2(x))
            return self.fc3(x)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Устройство: {device}")

    model = BreastCancerFFN(input_dim).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    #5. Обучение

    print("Начало обучения...")
    epochs = 50
    best_val_loss = float('inf')
    patience = 7
    patience_counter = 0
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)

        val_loss = val_loss / len(val_loader.dataset)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} — Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Ранняя остановка на эпохе {epoch+1}")
                break

    if best_model_state:
        model.load_state_dict(best_model_state)


    #6. Предсказание (ИСПРАВЛЕНА ОШИБКА ЗДЕСЬ)
    print("Генерация предсказаний...")
    model.eval()
    predictions_prob = []

    with torch.no_grad():
        for batch in test_loader:
            if isinstance(batch, (list, tuple)):
                X_batch = batch[0]
            else:
                X_batch = batch

            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            probs = torch.sigmoid(outputs)
            predictions_prob.extend(probs.cpu().numpy().flatten())

    predictions_class = (np.array(predictions_prob) >= 0.5).astype(int)

    result_df = pd.DataFrame({
        'patient_id': test_ids,
        'prediction': predictions_class
    })

    result_df.to_csv(output_filename, index=False)

    print(f"\nГотово! Файл '{output_filename}' сохранен.")
    print(result_df.head())

    from google.colab import files
    files.download(output_filename)

Проверка наличия файлов:
Train exists: True
Test exists: True
Файлы найдены! Начинаем работу...

Загрузка данных...
Train shape: (800, 19)
Test shape: (200, 18)
Предобработка...
Устройство: cpu
Начало обучения...
Epoch 10/50 — Train Loss: 0.1379, Val Loss: 0.1502
Epoch 20/50 — Train Loss: 0.0822, Val Loss: 0.1266
Ранняя остановка на эпохе 27
Генерация предсказаний...

Готово! Файл 'task1_predictions.csv' сохранен.
   patient_id  prediction
0           2           0
1           3           0
2           8           0
3          18           0
4          19           0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>